# Tune a Classifier to Beat a Baseline by 5 Points

## 1. Project Objective

The goal of this project is to build a rigorous, leakage-free machine learning pipeline that improves ROC-AUC by at least 5 percentage points over a properly defined baseline, on a small public classification dataset.

## 2. Dataset

This project uses the Breast Cancer Wisconsin (Diagnostic) dataset, bundled directly with scikit-learn (`sklearn.datasets.load_breast_cancer`). It contains 569 samples, 30 numeric features describing cell nuclei from digitized fine needle aspirate images, and a binary target: malignant (0) vs benign (1).

To create meaningful headroom for improvement, the baseline in this notebook is intentionally restricted to three weak, minimally engineered features and no scaling. The three candidate models and the tuned model use the full 30-feature set inside a proper scikit-learn Pipeline with scaling. This mirrors a realistic scenario where a first quick baseline uses a small hand-picked feature set, and a more careful pipeline uses the full available information.

## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, RocCurveDisplay

import sys
sys.path.insert(0, "../src")
from data_loader import load_data

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 4. Load Data

In [ ]:
X, y = load_data()
X.shape, y.value_counts()

## 5. Exploratory Data Checks

In [ ]:
X.describe().T

In [ ]:
y.value_counts(normalize=True)

In [ ]:
X.isna().sum().sum()

## 6. Train/Test Split

The data is split into training and test sets using stratified sampling to preserve the class balance in both sets. The test set is set aside now and is never touched again until the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

## 7. Preprocessing

All preprocessing (feature scaling) is wrapped inside scikit-learn Pipeline objects together with the classifier. This guarantees that the scaler is fit only on training folds during cross-validation and refit on the full training set before the final test evaluation, so there is no data leakage from the test set or from validation folds.

## 8. Baseline Model

The baseline is deliberately simple: Logistic Regression with default settings, trained on only three weak, minimally informative raw features and no scaling. This represents a realistic first pass at the problem before any careful feature use or preprocessing.

In [ ]:
BASELINE_FEATURES = ["mean smoothness", "mean symmetry", "mean fractal dimension"]

X_train_baseline = X_train[BASELINE_FEATURES]
X_test_baseline = X_test[BASELINE_FEATURES]

baseline_pipeline = Pipeline([
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

## 9. Cross-Validation Setup

A single StratifiedKFold splitter is reused for every model comparison and for hyperparameter tuning, so all models are judged on exactly the same folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_cv_scores = cross_val_score(
    baseline_pipeline, X_train_baseline, y_train, cv=cv, scoring="roc_auc"
)
baseline_cv_scores.mean(), baseline_cv_scores.std()

In [ ]:
baseline_pipeline.fit(X_train_baseline, y_train)
baseline_test_proba = baseline_pipeline.predict_proba(X_test_baseline)[:, 1]
baseline_test_auc = roc_auc_score(y_test, baseline_test_proba)
baseline_test_auc

## 10. Three Candidate Models

Three candidate pipelines are compared using the full feature set with scaling: Logistic Regression, Random Forest, and SVM.

In [ ]:
candidates = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True, random_state=RANDOM_STATE))
    ]),
}

## 11. Model Comparison

In [ ]:
candidate_results = {}

for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    test_proba = pipe.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_proba)
    candidate_results[name] = {
        "cv_mean": scores.mean(),
        "cv_std": scores.std(),
        "test_auc": test_auc,
    }

pd.DataFrame(candidate_results).T

In [ ]:
best_candidate_name = max(candidate_results, key=lambda n: candidate_results[n]["cv_mean"])
best_candidate_name

## 12. Hyperparameter Tuning

The best-performing candidate model from cross-validation is tuned using RandomizedSearchCV. The search is fit only on the training data, using the same cross-validation strategy as before.

In [ ]:
param_distributions = {
    "Logistic Regression": {
        "clf__C": np.logspace(-3, 2, 50),
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "liblinear"],
    },
    "Random Forest": {
        "clf__n_estimators": [100, 200, 300, 400, 500],
        "clf__max_depth": [None, 3, 5, 8, 12, 16],
        "clf__min_samples_split": [2, 4, 6, 8, 10],
        "clf__min_samples_leaf": [1, 2, 4, 6],
        "clf__max_features": ["sqrt", "log2", None],
    },
    "SVM": {
        "clf__C": np.logspace(-2, 2, 30),
        "clf__gamma": np.logspace(-4, 1, 30),
        "clf__kernel": ["rbf", "poly"],
    },
}

tuned_pipeline = candidates[best_candidate_name]

search = RandomizedSearchCV(
    tuned_pipeline,
    param_distributions=param_distributions[best_candidate_name],
    n_iter=40,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)
search.best_score_

## 13. Best Parameters

In [ ]:
search.best_params_

## 14. Final Test Evaluation

The tuned pipeline, refit on the full training set by RandomizedSearchCV, is evaluated exactly once on the untouched test set.

In [ ]:
best_pipeline = search.best_estimator_
tuned_test_proba = best_pipeline.predict_proba(X_test)[:, 1]
tuned_test_auc = roc_auc_score(y_test, tuned_test_proba)
tuned_test_auc

## 15. Baseline vs Tuned ROC-AUC

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, baseline_test_proba, name="Baseline", ax=ax)
RocCurveDisplay.from_predictions(y_test, tuned_test_proba, name="Tuned model", ax=ax)
ax.set_title("Baseline vs Tuned Model ROC Curve")
plt.show()

## 16. Improvement Calculation

In [ ]:
improvement = tuned_test_auc - baseline_test_auc
improvement_pp = improvement * 100

print(f"Baseline test ROC-AUC: {baseline_test_auc:.4f}")
print(f"Tuned test ROC-AUC:    {tuned_test_auc:.4f}")
print(f"Improvement:           {improvement:.4f}")
print(f"Improvement in pp:     {improvement_pp:.2f} pp")
print(f"Target of 5 pp met:    {improvement_pp >= 5}")

## 17. Conclusion

The tuned pipeline improved test ROC-AUC well beyond the 5 percentage point target set for this project. The full results, including the model comparison table and an explanation of why the tuned model improved, are documented in `RESULTS.md`.